# 📊 VIGIL — Dataset Exploration & Model Benchmarking
Deep-dive into the UCF-Crime dataset, visualise frame distributions, and benchmark detection modules.

In [ ]:
import kagglehub, os, cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
from pathlib import Path
import sys
sys.path.append('..')

path = kagglehub.dataset_download('odins0n/ucf-crime-dataset')
print(f'Dataset: {path}')
DATASET_PATH = path

## 1. Category Distribution

In [ ]:
CATEGORIES = [
    'Abuse','Arrest','Arson','Assault','Burglary','Explosion',
    'Fighting','RoadAccidents','Robbery','Shooting','Shoplifting',
    'Stealing','Vandalism','Normal_Videos'
]
counts = defaultdict(int)
for root, _, files in os.walk(DATASET_PATH):
    for cat in CATEGORIES:
        if cat in root:
            counts[cat] += sum(1 for f in files if f.endswith(('.mp4','.avi')))

cats   = sorted(counts, key=counts.get, reverse=True)
values = [counts[c] for c in cats]
colors = ['#e8453c' if c != 'Normal_Videos' else '#00c9b1' for c in cats]

plt.figure(figsize=(14,5))
bars = plt.bar(cats, values, color=colors, edgecolor='none')
plt.xticks(rotation=35, ha='right')
plt.ylabel('Number of Videos')
plt.title('UCF-Crime — Videos per Category', fontsize=14, fontweight='bold')
for bar, val in zip(bars, values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150)
plt.show()

## 2. Frame Comparison — Normal vs Anomalous

In [ ]:
def sample_frame(directory, category):
    for root, _, files in os.walk(directory):
        if category not in root:
            continue
        for f in files:
            if f.endswith(('.mp4','.avi')):
                cap = cv2.VideoCapture(os.path.join(root,f))
                cap.set(cv2.CAP_PROP_POS_FRAMES, 50)
                ret, fr = cap.read()
                cap.release()
                if ret:
                    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
    return None

anomalous_cats = ['Fighting','Burglary','Robbery','Shooting','Vandalism','Arson']
fig, axes = plt.subplots(2, len(anomalous_cats)//2 + 1, figsize=(18,8))
axes = axes.flat

# Normal sample
nf = sample_frame(DATASET_PATH, 'Normal_Videos')
if nf is not None:
    next(axes).imshow(nf); next(axes).set_title('Normal', color='green', fontweight='bold')

for cat in anomalous_cats:
    fr = sample_frame(DATASET_PATH, cat)
    ax = next(axes, None)
    if ax and fr is not None:
        ax.imshow(fr)
        ax.set_title(cat, color='red', fontweight='bold')

for ax in axes:
    ax.axis('off')

plt.suptitle('Frame Comparison: Normal vs Anomalous Categories', fontsize=14)
plt.tight_layout()
plt.savefig('frame_comparison.png', dpi=150)
plt.show()

## 3. Motion Score Distribution

In [ ]:
from src.preprocessing.frame_processor       import FrameProcessor
from src.preprocessing.background_subtractor import BackgroundSubtractor

proc = FrameProcessor(target_size=(320,180), fps=3)
sub  = BackgroundSubtractor()

motion_data = defaultdict(list)
sample_cats = {'Normal_Videos':2, 'Fighting':2, 'Burglary':2}

for cat, max_vids in sample_cats.items():
    vid_count = 0
    for root, _, files in os.walk(DATASET_PATH):
        if cat not in root or vid_count >= max_vids:
            continue
        for f in files:
            if not f.endswith(('.mp4','.avi')):
                continue
            frames = proc.extract_frames(os.path.join(root,f), max_frames=30)
            scores = sub.process_sequence(frames)
            motion_data[cat].extend(scores)
            vid_count += 1
            if vid_count >= max_vids:
                break

plt.figure(figsize=(12,5))
palette = {'Normal_Videos':'#00c9b1','Fighting':'#e8453c','Burglary':'#f5a623'}
for cat, scores in motion_data.items():
    if scores:
        plt.hist(scores, bins=30, alpha=0.6, color=palette.get(cat,'gray'),
                 label=cat, edgecolor='none')
plt.xlabel('Motion Score')
plt.ylabel('Frequency')
plt.title('Motion Score Distribution by Category')
plt.legend()
plt.grid(True, alpha=0.2)
plt.savefig('motion_distribution.png', dpi=150)
plt.show()

## 4. YOLOv8 Detection Demo on Multiple Categories

In [ ]:
from src.detection.object_detector import ObjectDetector

det = ObjectDetector(model_size='n', confidence=0.35)
demo_cats = ['Fighting','Burglary','Normal_Videos']

fig, axes = plt.subplots(1, len(demo_cats), figsize=(18,6))
for ax, cat in zip(axes, demo_cats):
    fr = None
    for root, _, files in os.walk(DATASET_PATH):
        if cat not in root: continue
        for f in files:
            if f.endswith(('.mp4','.avi')):
                cap = cv2.VideoCapture(os.path.join(root,f))
                cap.set(cv2.CAP_PROP_POS_FRAMES, 80)
                ret, fr = cap.read()
                cap.release()
                if ret: break
        if fr is not None: break

    if fr is not None:
        dets      = det.detect(fr)
        annotated = det.draw_detections(fr.copy(), dets)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{cat}\n({len(dets)} detections)', fontweight='bold')
    ax.axis('off')

plt.suptitle('YOLOv8 Object Detection — Cross-Category Demo', fontsize=13)
plt.tight_layout()
plt.savefig('yolo_detections.png', dpi=150)
plt.show()

## 5. Optical Flow Visualisation

In [ ]:
from src.preprocessing.optical_flow import OpticalFlowExtractor

extractor = OpticalFlowExtractor(method='farneback')

demo_cats = ['Fighting','Normal_Videos']
fig, axes = plt.subplots(2, 2, figsize=(14,8))

for col, cat in enumerate(demo_cats):
    frames = []
    for root, _, files in os.walk(DATASET_PATH):
        if cat not in root: continue
        for f in files:
            if f.endswith(('.mp4','.avi')):
                p   = FrameProcessor(target_size=(320,180), fps=10, normalize=False)
                frs = p.extract_frames(os.path.join(root,f), max_frames=3)
                frames.extend(frs)
                if len(frames) >= 2: break
        if len(frames) >= 2: break

    if len(frames) >= 2:
        flow = extractor.compute_flow(frames[0], frames[1])
        vis  = extractor.visualize_flow(flow)
        axes[0,col].imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB))
        axes[0,col].set_title(f'{cat} — Frame', fontweight='bold')
        axes[1,col].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        axes[1,col].set_title(f'{cat} — Optical Flow')
    for r in range(2):
        axes[r,col].axis('off')

plt.suptitle('Optical Flow: Normal vs Fighting', fontsize=13)
plt.tight_layout()
plt.savefig('optical_flow_comparison.png', dpi=150)
plt.show()